# Querying Point Signed Distance Fields (SDF)
This example highlights computing exact Signed Distance Fields (SDF) and projecting points onto a 3D mesh.

## SDF Computation
We generate a dense point cloud or volumetric grid of query points and utilize Conquer3D to evaluate the SDF for each point. The example covers distance calculations, finding the closest points on the mesh surface, and comparing performance against standard libraries.

In [1]:
import open3d as o3d
import torch
import conquer3d as c3d
import plotly.graph_objects as go
import numpy as np
import random

# 1. Create a manifold sphere mesh
sphere = o3d.geometry.TriangleMesh.create_sphere(radius=1.0, resolution=10)
sphere.compute_vertex_normals()
vertices = torch.tensor(np.asarray(sphere.vertices), dtype=torch.float32, device='cuda')
triangles = torch.tensor(np.asarray(sphere.triangles), dtype=torch.int32, device='cuda')

# 2. Build the TriangleMesh
mesh = c3d.data_structure.TriangleMesh(vertices, triangles)

# 3. Sample 5 inside points and 5 outside points
inside_pts = []
for _ in range(5):
    # Inside radius < 1.0
    r = random.uniform(0.1, 0.8)
    theta = random.uniform(0, 2*np.pi)
    phi = random.uniform(0, np.pi)
    x = r * np.sin(phi) * np.cos(theta)
    y = r * np.sin(phi) * np.sin(theta)
    z = r * np.cos(phi)
    inside_pts.append([x,y,z])
    
outside_pts = []
for _ in range(5):
    # Outside radius > 1.0
    r = random.uniform(1.2, 2.0)
    theta = random.uniform(0, 2*np.pi)
    phi = random.uniform(0, np.pi)
    x = r * np.sin(phi) * np.cos(theta)
    y = r * np.sin(phi) * np.sin(theta)
    z = r * np.cos(phi)
    outside_pts.append([x,y,z])

query_points_np = np.array(inside_pts + outside_pts)
query_points = torch.tensor(query_points_np, dtype=torch.float32, device='cuda')

# 4. Query points with SDF and projected points
q_ids, tri_ids, prj_pts, sdf = mesh.query_points(
    query_points, 
    return_sdf=True, 
    return_prj_pts=True, 
    sign_mode=0,
    distance_mode=0
)

prj_pts_np = prj_pts.cpu().numpy()
sdf_np = sdf.cpu().numpy()

# 5. Visualize
fig = go.Figure()

# Plot base mesh
fig.add_trace(go.Mesh3d(
    x=vertices[:, 0].cpu(), y=vertices[:, 1].cpu(), z=vertices[:, 2].cpu(),
    i=triangles[:, 0].cpu(), j=triangles[:, 1].cpu(), k=triangles[:, 2].cpu(),
    color='lightblue', opacity=0.3, name='Base Mesh'
))

# Extract edges for wireframe
verts_np = vertices.cpu().numpy()
tris_np = triangles.cpu().numpy()
edge_x = []
edge_y = []
edge_z = []
for tri in tris_np:
    v0, v1, v2 = verts_np[tri[0]], verts_np[tri[1]], verts_np[tri[2]]
    edge_x.extend([v0[0], v1[0], v2[0], v0[0], None])
    edge_y.extend([v0[1], v1[1], v2[1], v0[1], None])
    edge_z.extend([v0[2], v1[2], v2[2], v0[2], None])

fig.add_trace(go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode='lines',
    line=dict(color='gray', width=1),
    name='Wireframe',
    showlegend=False
))

colors = ['red', 'green', 'blue', 'orange', 'purple', 'cyan', 'magenta', 'yellow', 'brown', 'pink']

# Plot pairs of points and text for SDF
for i in range(10):
    qp = query_points_np[i]
    pp = prj_pts_np[i]
    dist = sdf_np[i]
    c = colors[i]
    
    # Query point
    fig.add_trace(go.Scatter3d(
        x=[qp[0]], y=[qp[1]], z=[qp[2]],
        mode='markers+text',
        marker=dict(size=5, color=c),
        text=[f'SDF: {dist:.3f}'],
        textposition='top center',
        name=f'Query {i}'
    ))
    
    # Projected point
    fig.add_trace(go.Scatter3d(
        x=[pp[0]], y=[pp[1]], z=[pp[2]],
        mode='markers',
        marker=dict(size=4, color=c, symbol='diamond'),
        name=f'Proj {i}'
    ))
    
    # Line connecting them
    fig.add_trace(go.Scatter3d(
        x=[qp[0], pp[0]], y=[qp[1], pp[1]], z=[qp[2], pp[2]],
        mode='lines',
        line=dict(color=c, width=3),
        showlegend=False
    ))

fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False)
    ),
    title="Point-to-Mesh Projection and SDF"
)
fig.show()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
import time

# 1. Create a dense 3D grid (e.g., 64x64x64 = 262,144 points)
res = 64
grid_min, grid_max = -1.5, 1.5
x = torch.linspace(grid_min, grid_max, res, device='cuda')
y = torch.linspace(grid_min, grid_max, res, device='cuda')
z = torch.linspace(grid_min, grid_max, res, device='cuda')
grid_x, grid_y, grid_z = torch.meshgrid(x, y, z, indexing='ij')
query_points_dense = torch.stack([grid_x.flatten(), grid_y.flatten(), grid_z.flatten()], dim=1).contiguous()

print(f"Benchmarking SDF query for {query_points_dense.shape[0]} points...")

# ---------------------------------------------------------
# Conquer3D (c3d) Evaluation
# ---------------------------------------------------------
torch.cuda.synchronize()
start_c3d = time.time()
# sign_mode=0 ensures ray casting sign calculation
_, _, _, sdf_c3d = mesh.query_points(query_points_dense, return_sdf=True, sign_mode=0)
torch.cuda.synchronize()
time_c3d = time.time() - start_c3d
print(f"[Conquer3D] Time: {time_c3d:.5f} seconds")

# ---------------------------------------------------------
# Open3D Evaluation
# ---------------------------------------------------------
# Setup Open3D Scene (Requires CPU for RaycastingScene by default)
mesh_t = o3d.t.geometry.TriangleMesh.from_legacy(sphere).to(o3d.core.Device('CPU:0'))
scene = o3d.t.geometry.RaycastingScene()
scene.add_triangles(mesh_t)

# Convert points to Open3D CPU Tensor
query_points_cpu = query_points_dense.cpu()
query_points_o3d = o3d.core.Tensor.from_dlpack(torch.utils.dlpack.to_dlpack(query_points_cpu))

torch.cuda.synchronize()
start_o3d = time.time()
sdf_o3d_tensor = scene.compute_signed_distance(query_points_o3d)
torch.cuda.synchronize()
time_o3d = time.time() - start_o3d
print(f"[Open3D]    Time: {time_o3d:.5f} seconds")

# ---------------------------------------------------------
# Accuracy Comparison
# ---------------------------------------------------------
sdf_o3d = torch.utils.dlpack.from_dlpack(sdf_o3d_tensor.to_dlpack()).cuda()

diff = torch.abs(sdf_c3d - sdf_o3d)
max_diff = diff.max().item()
mean_diff = diff.mean().item()

print("\n--- Comparison ---")
print(f"Max Difference:  {max_diff:.8f}")
print(f"Mean Difference: {mean_diff:.8f}")


Benchmarking SDF query for 262144 points...
[Conquer3D] Time: 0.00076 seconds
[Open3D]    Time: 0.01582 seconds

--- Comparison ---
Max Difference:  0.00000024
Mean Difference: 0.00000001
